- https://github.com/verl-project/verl-recipe/tree/main/retool
- 数据，算法，训练
    - 数据即任务；
    - 算法：sft/rl, sft: ntp, rl: online rollout interaction (reward);
    - 训练：一定是分布式的（单机多卡/多机多卡），积累性能调优经验；
- 本期内容
    - tool use data curation
    - multiturn sft：loss mask
    - 分布式训练的一些参数配置
        - 显存分析
        - （torchrun，fsdp，sp: 序列并行）

### data curation

- 将普通的数学推理数据 $\mathcal D_{\text{init}}$ 自动升级为代码增强数据 $\mathcal D_{CI}$
    - 数据采集与清洗: 收集 OpenThoughts 等高质量数据（Text-based CoT）。使用“专家人工” + “DeepSeek-R1”双重过滤，剔除无效数据。
    - 自动化 Prompt 转换: 利用特定模板（见右侧示例），将文本中的手动计算步骤，替换为 Python 代码块和执行结果，保留原始推理逻辑。
    - 双重验证 (Dual-Verification):
        - 格式验证：确保代码语法正确，标签闭合。
        - 答案验证：确保执行后的最终答案与 Ground Truth 一致。
- https://huggingface.co/datasets/JoeYing/ReTool-SFT
    - 2k high quality sft is enough (qwen2.5-32b-instruct)
  
```markdown
Transformation Prompt Logic (Simplified)
User Prompt:
...
Original Thinking Process:
<original_thinking_process>
1. First, calculate 25 * 43.
2. The result is 1075.
...
</original_thinking_process>

Goal:
Replace manual calculation steps with code snippets.
Keep logic intact.

Target Output (Revised):
<revised_thinking_process>
Let's calculate 25 * 43 using code.
<code>
print(25 * 43)
</code>
<interpreter>1075</interpreter>
...
</revised_thinking_process>
```

### data preprocess

```sh
# 下载并安装 verl
git clone https://github.com/volcengine/verl.git
git clone git@github.com:verl-project/verl-recipe.git

# dataset
# https://huggingface.co/datasets/JoeYing/ReTool-SFT
# 构建跟 ci 交互的多轮对话
python3 recipe/retool/retool_sft_preprocess.py 

DatasetDict({
    train: Dataset({
        features: ['messages', 'tools'],
        num_rows: 2000
    })
})
```

- 2000 条
    - 2000/32 * 6
- dataset sample
    - https://gemini.google.com/app/b47f2bad0728c0c2

In [5]:
(2000/32), 62 * 6

(62.5, 372)

In [6]:
raw_data = [
{
"content": "Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (wrapped in `<interpreter>output_str</interpreter>`) can be returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports. \nEach code snippet is wrapped with `<code>\n```python\ncode snippet\n```\n</code>`.\nThe last part of your response should be in the following format:\n<answer>\n\\boxed{{'The final answer goes here.'}}\n</answer>\n\n*user question:*\nThere are 152 students at Dala High School. Assume the following: \n- 100 students take a Math class \n- 94 students take a Science class \n- 57 students take an English class \n- 73 students take a Math class and a Science class \n- 24 students take a Math class and an English class \n- 27 students take a Science class and an English class \n- 22 students take a Math class and a Science class and an English class\n \nHow many students take neither a Math class nor a Science class nor an Eglish class?\n\nRemember to place the final answer in the last part using the format: \n<answer>\n\\boxed{{'The final answer goes here.'}}\n</answer>",
"role": "user"
},
{
"content": "Okay, so I need to find out how many students at Dala High School are not taking any of the three classes: Math, Science, or English. The total number of students is 152. Let me see... I remember this is a problem about sets and maybe using the principle of inclusion-exclusion. Let me recall how that works.\n\nFirst, the inclusion-exclusion principle for three sets says that the total number of students taking at least one of the classes is equal to the sum of the numbers in each individual class, minus the sum of the numbers in each pair of classes, plus the number in all three classes. Then, subtract that total from the overall number of students to get those not taking any of the three. \n\nLet me write that down step by step. Let M be the set of students taking Math, S for Science, and E for English. Then:\n\nTotal in at least one class = |M ∪ S ∪ E| = |M| + |S| + |E| - |M ∩ S| - |M ∩ E| - |S ∩ E| + |M ∩ S ∩ E|\n\nGiven the numbers:\n\n|M| = 100\n\n|S| = 94\n\n|E| = 57\n\n|M ∩ S| = 73\n\n|M ∩ E| = 24\n\n|S ∩ E| = 27\n\n|M ∩ S ∩ E| = 22\n\nSo plugging these into the formula:\n\nTotal = 100 + 94 + 57 - 73 - 24 - 27 + 22\n\nLet me compute that step by step using code to ensure accuracy.\n\n<code>\n```python\n\nM = 100\nS = 94\nE = 57\nM_S = 73\nM_E = 24\nS_E = 27\nM_S_E = 22\n\ntotal_in_any = M + S + E - M_S - M_E - S_E + M_S_E\nstudents_neither = 152 - total_in_any\nprint(students_neither)\n```\n</code>\n\n<interpreter>\n3\n</interpreter>\n\nThe code returns 3, indicating that 3 students take none of the classes. To verify, I'll break down each category to ensure the total matches:\n\n- Only Math and Science: 73 - 22 = 51\n- Only Math and English: 24 - 22 = 2\n- Only Science and English: 27 - 22 = 5\n- Only Math: 100 - 51 - 2 - 22 = 25\n- Only Science: 94 - 51 - 5 - 22 = 16\n- Only English: 57 - 2 - 5 - 22 = 28\n\nSumming all categories:\n\n<code>\n```python\n\nonly_M_S = 73 - 22\nonly_M_E = 24 - 22\nonly_S_E = 27 - 22\n\nonly_M = 100 - only_M_S - only_M_E - 22\nonly_S = 94 - only_M_S - only_S_E - 22\nonly_E = 57 - only_M_E - only_S_E - 22\n\ntotal_verify = only_M + only_S + only_E + only_M_S + only_M_E + only_S_E + 22\nprint(total_verify)\n```\n</code>\n\n<interpreter>\n149\n</interpreter>\n\nThe verification total is 149, so students not taking any classes are 152 - 149 = 3, confirming the initial result.\n\n<answer>\n\\boxed{3}\n</answer>",
"role": "assistant"
}
]

In [7]:
len(raw_data)

2

In [8]:
print(raw_data[0]['content'])

Solve the following problem step by step. You now have the ability to selectively write executable Python code to enhance your reasoning process. The Python code will be executed by an external sandbox, and the output (wrapped in `<interpreter>output_str</interpreter>`) can be returned to aid your reasoning and help you arrive at the final answer. The Python code should be complete scripts, including necessary imports. 
Each code snippet is wrapped with `<code>
```python
code snippet
```
</code>`.
The last part of your response should be in the following format:
<answer>
\boxed{{'The final answer goes here.'}}
</answer>

*user question:*
There are 152 students at Dala High School. Assume the following: 
- 100 students take a Math class 
- 94 students take a Science class 
- 57 students take an English class 
- 73 students take a Math class and a Science class 
- 24 students take a Math class and an English class 
- 27 students take a Science class and an English class 
- 22 students ta

In [9]:
print(raw_data[1]['content'])

Okay, so I need to find out how many students at Dala High School are not taking any of the three classes: Math, Science, or English. The total number of students is 152. Let me see... I remember this is a problem about sets and maybe using the principle of inclusion-exclusion. Let me recall how that works.

First, the inclusion-exclusion principle for three sets says that the total number of students taking at least one of the classes is equal to the sum of the numbers in each individual class, minus the sum of the numbers in each pair of classes, plus the number in all three classes. Then, subtract that total from the overall number of students to get those not taking any of the three. 

Let me write that down step by step. Let M be the set of students taking Math, S for Science, and E for English. Then:

Total in at least one class = |M ∪ S ∪ E| = |M| + |S| + |E| - |M ∩ S| - |M ∩ E| - |S ∩ E| + |M ∩ S ∩ E|

Given the numbers:

|M| = 100

|S| = 94

|E| = 57

|M ∩ S| = 73

|M ∩ E| = 2

In [10]:
M = 100
S = 94
E = 57
M_S = 73
M_E = 24
S_E = 27
M_S_E = 22

total_in_any = M + S + E - M_S - M_E - S_E + M_S_E
students_neither = 152 - total_in_any
print(students_neither)

3


In [11]:
only_M_S = 73 - 22
only_M_E = 24 - 22
only_S_E = 27 - 22

only_M = 100 - only_M_S - only_M_E - 22
only_S = 94 - only_M_S - only_S_E - 22
only_E = 57 - only_M_E - only_S_E - 22

total_verify = only_M + only_S + only_E + only_M_S + only_M_E + only_S_E + 22
print(total_verify)

149


In [12]:
sft_data = {'messages': [{'content': "There are 152 students at Dala High School. Assume the following:  \n- 100 students take a Math class  \n- 94 students take a Science class  \n- 57 students take an English class  \n- 73 students take a Math class and a Science class  \n- 24 students take a Math class and an English class  \n- 27 students take a Science class and an English class  \n- 22 students take a Math class and a Science class and an English class\n  \nHow many students take neither a Math class nor a Science class nor an Eglish class?\n\nRemember to place the final answer in the last part using the format: \n\n\\boxed{{'The final answer goes here.'}}",
   'role': 'user',
   'tool_calls': None},
  {'content': 'Okay, so I need to find out how many students at Dala High School are not taking any of the three classes: Math, Science, or English. The total number of students is 152. Let me see... I remember this is a problem about sets and maybe using the principle of inclusion-exclusion. Let me recall how that works.\n\nFirst, the inclusion-exclusion principle for three sets says that the total number of students taking at least one of the classes is equal to the sum of the numbers in each individual class, minus the sum of the numbers in each pair of classes, plus the number in all three classes. Then, subtract that total from the overall number of students to get those not taking any of the three. \n\nLet me write that down step by step. Let M be the set of students taking Math, S for Science, and E for English. Then:\n\nTotal in at least one class = |M ∪ S ∪ E| = |M| + |S| + |E| - |M ∩ S| - |M ∩ E| - |S ∩ E| + |M ∩ S ∩ E|\n\nGiven the numbers:\n\n|M| = 100\n\n|S| = 94\n\n|E| = 57\n\n|M ∩ S| = 73\n\n|M ∩ E| = 24\n\n|S ∩ E| = 27\n\n|M ∩ S ∩ E| = 22\n\nSo plugging these into the formula:\n\nTotal = 100 + 94 + 57 - 73 - 24 - 27 + 22\n\nLet me compute that step by step using code to ensure accuracy.',
   'role': 'assistant',
   'tool_calls': [{'function': {'arguments': {'code': 'M = 100\nS = 94\nE = 57\nM_S = 73\nM_E = 24\nS_E = 27\nM_S_E = 22\n\ntotal_in_any = M + S + E - M_S - M_E - S_E + M_S_E\nstudents_neither = 152 - total_in_any\nprint(students_neither)'},
      'name': 'code_interpreter'},
     'type': 'function'}]},
  {'content': '3', 'role': 'tool', 'tool_calls': None},
  {'content': "The code returns 3, indicating that 3 students take none of the classes. To verify, I'll break down each category to ensure the total matches:\n\n- Only Math and Science: 73 - 22 = 51\n- Only Math and English: 24 - 22 = 2\n- Only Science and English: 27 - 22 = 5\n- Only Math: 100 - 51 - 2 - 22 = 25\n- Only Science: 94 - 51 - 5 - 22 = 16\n- Only English: 57 - 2 - 5 - 22 = 28\n\nSumming all categories:",
   'role': 'assistant',
   'tool_calls': [{'function': {'arguments': {'code': 'only_M_S = 73 - 22\nonly_M_E = 24 - 22\nonly_S_E = 27 - 22\n\nonly_M = 100 - only_M_S - only_M_E - 22\nonly_S = 94 - only_M_S - only_S_E - 22\nonly_E = 57 - only_M_E - only_S_E - 22\n\ntotal_verify = only_M + only_S + only_E + only_M_S + only_M_E + only_S_E + 22\nprint(total_verify)'},
      'name': 'code_interpreter'},
     'type': 'function'}]},
  {'content': '149', 'role': 'tool', 'tool_calls': None},
  {'content': 'The verification total is 149, so students not taking any classes are 152 - 149 = 3, confirming the initial result.\n\n\n\\boxed{3}',
   'role': 'assistant',
   'tool_calls': None}],
 'tools': [{'function': {'description': 'A tool for executing code.',
    'name': 'code_interpreter',
    'parameters': {'properties': {'code': {'description': 'The code to execute.',
       'type': 'string'}},
     'required': ['code'],
     'type': 'object'}},
   'type': 'function'}]}

In [13]:
sft_data.keys()

dict_keys(['messages', 'tools'])

In [14]:
# user -> assistant (content w/ tool calls) -> tool (tool result) -> assistant (content w/ tool calls) -> tool (result) -> assistant (content)
sft_data['messages']

[{'content': "There are 152 students at Dala High School. Assume the following:  \n- 100 students take a Math class  \n- 94 students take a Science class  \n- 57 students take an English class  \n- 73 students take a Math class and a Science class  \n- 24 students take a Math class and an English class  \n- 27 students take a Science class and an English class  \n- 22 students take a Math class and a Science class and an English class\n  \nHow many students take neither a Math class nor a Science class nor an Eglish class?\n\nRemember to place the final answer in the last part using the format: \n\n\\boxed{{'The final answer goes here.'}}",
  'role': 'user',
  'tool_calls': None},
 {'content': 'Okay, so I need to find out how many students at Dala High School are not taking any of the three classes: Math, Science, or English. The total number of students is 152. Let me see... I remember this is a problem about sets and maybe using the principle of inclusion-exclusion. Let me recall how

In [15]:
from rich.pretty import pprint
pprint(sft_data['tools'])

[
│   {
│   │   'function': {
│   │   │   'description': 'A tool for executing code.',
│   │   │   'name': 'code_interpreter',
│   │   │   'parameters': {
│   │   │   │   'properties': {'code': {'description': 'The code to execute.', 'type': 'string'}},
│   │   │   │   'required': ['code'],
│   │   │   │   'type': 'object'
│   │   │   }
│   │   },
│   │   'type': 'function'
│   }
]

### MultiTurnSFT


```
data.train_batch_size=32 \
data.multiturn.enable=true \
data.multiturn.messages_key=messages \
data.multiturn.tools_key=tools \
```
- MultiTurnSFTDataset 核心是处理和消化 loss mask
    - Dataset for multi-turn conversations where each assistant response should be trained
    - system, user, tool: loss masked
```python
def __getitem__(self, ):
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "position_ids": position_ids,
        "loss_mask": loss_mask,
    }
```
- loss mask 为 1（计算 loss）
    - assistant 的输出: reasoning, tool_call (name, parameters), `<|im_end|>`

In [21]:
import torch
import sys
import os
from transformers import AutoTokenizer
from verl.utils.dataset.multiturn_sft_dataset import MultiTurnSFTDataset
from omegaconf import OmegaConf

MODEL_PATH = "Qwen/Qwen2.5-7B-Instruct"  # 你的模型路径
DATA_PATH = "./data/train-00000-of-00001.parquet" # 你的数据路径
MAX_SAMPLES = 3 # 检查前几个样本

class Color:
    GREEN = '\033[92m' # Loss Mask = 1
    GRAY = '\033[90m'  # Loss Mask = 0
    RESET = '\033[0m'

def visualize_dataset_mask():
    print(f"Loading tokenizer from {MODEL_PATH}...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    
    # 构造简单的 Config
    config = OmegaConf.create({
        "pad_mode": "right",
        "max_length": 5120,
        "truncation": "error",
        "multiturn": {
            "enable": True,
            "messages_key": "messages",
            "tools_key": "tools",
        }
    })

    print(f"Loading dataset from {DATA_PATH}...")
    dataset = MultiTurnSFTDataset(
        parquet_files=[DATA_PATH],
        tokenizer=tokenizer,
        config=config,
        max_samples=MAX_SAMPLES
    )

    print(f"\n{'='*20} Checking {len(dataset)} Samples {'='*20}\n")

    for i in range(len(dataset)):
        # __getitem__
        item = dataset[i]
        input_ids = item['input_ids']
        loss_mask = item['loss_mask']
        
        # 确保是 tensor 并且在 CPU 上
        if isinstance(input_ids, torch.Tensor):
            input_ids = input_ids.tolist()
        if isinstance(loss_mask, torch.Tensor):
            loss_mask = loss_mask.tolist()

        print(f"Sample {i}:")
        print("-" * 50)
        
        # 逐个 Token 解码并根据 Mask 染色
        output_text = ""
        current_mask = -1 # 用于记录上一个 mask 状态，减少颜色控制符数量
        
        for tid, mask in zip(input_ids, loss_mask):
            # 跳过 Padding (通常是 0 或 tokenizer.pad_token_id)
            if tid == tokenizer.pad_token_id and mask == 0:
                continue

            token_text = tokenizer.decode([tid])
            
            # 颜色切换逻辑
            if mask != current_mask:
                if mask == 1:
                    output_text += Color.RESET + Color.GREEN
                else:
                    output_text += Color.RESET + Color.GRAY
                current_mask = mask
            
            output_text += token_text

        print(output_text + Color.RESET)
        print("-" * 50)
        
        # 统计信息
        valid_tokens = sum(loss_mask)
        total_tokens = len([x for x in input_ids if x != tokenizer.pad_token_id])
        print(f"Valid Training Tokens (Mask=1): {valid_tokens} / {total_tokens} (Total Non-Pad)")
        print("\n")

In [22]:
visualize_dataset_mask()

Loading tokenizer from Qwen/Qwen2.5-7B-Instruct...
Loading dataset from ./data/train-00000-of-00001.parquet...
dataset len: 2000
selected 5 random samples out of 2000

==================== Checking 5 Samples ====================

Sample 0:
--------------------------------------------------
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"function": {"description": "A tool for executing code.", "name": "code_interpreter", "parameters": {"properties": {"code": {"description": "The code to execute.", "type": "string"}}, "required": ["code"], "type": "object"}}, "type": "function"}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call><|

### training & evaluation

- 显存分析
    - model weights：7B * 2 = 14GB
        - fsdp 4卡分片：14/4 = 3.5GB
    - AdamW 优化器需要存储 momentum 和 variance（fp32）：
        -  7B × 4  × 2 / 4 = 21GB (1阶/2阶)
        -  7B * 4 / 4 = 7GB (gradients)
    - 中间激活层（是否开启 gradient checkpointing）
        - 检查的数量：$\sqrt{28}=6$
        - embedding dim: 3584
        - seq len * embed dim * n * 2
    - logits: (micro_batch_size, seq_len/sp_size, vocab_size)，非常耗费显存
        - 4 * 16384/2 * 152064 * 2 = 9.28GB
- `sft_trainer.yaml`:
    - 默认开启 enable_gradient_checkpointing

- `verl/trainer/fsdp_sft_trainer.py`
    - `dp_size = world_size // config.ulysses_sequence_parallel_size`
- `data.train_batch_size`: 是全局的 Global batch size（假如是32）
    - $\text{Global Batch Size} = \text{DP Size} \times \text{Batch Size per DP Rank}$
    - dp size = 4/2 = 2
    - 每个 dp 组，是 32 / 2 = 16
    - 考虑到 ga（`data.micro_batch_size_per_gpu=4`）
        - $ \text{Gradient Accumulation Steps} = \frac{\text{Per DP Batch Size}}{\text{Micro Batch Size}} = \frac{16}{4} = 4 $
        - 这意味着每做一次优化器更新 (optimizer step)，模型会连续前向/反向传播 4 次微批次。

- training process：能数据并行就数据并行，其它能不分布式就不分布式；
    - 4卡，7b，2000/32 * 6，1小时可以搞定；
    - 每一个 epoch 结束，下一个 epoch开启，loss 都会有一个陡降；
        - 每一个 epoch 保存一个 ckpt
- model merger (fsdp weights to hf weights)
    - https://github.com/volcengine/verl/blob/main/scripts/legacy_model_merger.py
    ```
    python scripts/legacy_model_merger.py merge \
        --backend fsdp \
        --local_dir /xxx/multiturn-sft-qwen-2.5-7b-instruct-1204/global_step_372 \
        --target_dir /xx/multiturn-sft-qwen-2.5-7b-instruct-1204/weights
    ```